In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

In [3]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='object')

In [4]:
columns = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]

df = df[columns]

In [5]:
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [6]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [7]:
df['horsepower'].median()

254.0

In [8]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

In [9]:
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [10]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

In [11]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [12]:
df_train.head()

,engine_displacement,horsepower,vehicle_weight,model_year
6252,1900,239.0,3790,2015
4684,2000,224.0,4380,1992
1731,2330,286.0,4090,2022
4742,2300,NaN,4150,1997
4521,2340,269.0,4400,2001


In [13]:
mean_hp = df_train['horsepower'].mean()
mean_hp

np.float64(254.46338337605272)

In [14]:
#filling with 0 

X_train_zero = df_train.fillna(0).values
X_val_zero = df_val.fillna(0).values

In [15]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv @ X.T @ y

    return w_full[0], w_full[1:]

In [16]:
w0_zero, w_zero = train_linear_regression(X_train_zero, y_train)

In [17]:
y_pred_zero = w0_zero + X_val_zero @ w_zero

In [18]:
def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [19]:
score_zero = rmse(y_val, y_pred_zero)
round(score_zero, 3)

np.float64(2.205)

In [20]:
#Fill with mean

X_train_mean = df_train.fillna(mean_hp).values
X_val_mean = df_val.fillna(mean_hp).values

In [21]:
w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

y_pred_mean = w0_mean + X_val_mean @ w_mean

In [22]:
score_mean = rmse(y_val, y_pred_mean)
round(score_mean, 3)

np.float64(2.202)

In [23]:
#Lower RMSE is better, so better option is with mean

In [24]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T @ X
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv @ X.T @ y

    return w_full[0], w_full[1:]

In [25]:
X_train_zero = df_train.fillna(0).values
X_val_zero = df_val.fillna(0).values

In [26]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train_zero, y_train, r=r)
    y_pred = w0 + X_val_zero @ w
    score = rmse(y_val, y_pred)
    print(r, round(score, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [27]:
scores = []

for seed in range(10):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train_seed = df.iloc[idx[:n_train]].copy()
    df_val_seed = df.iloc[idx[n_train:n_train + n_val]].copy()

    y_train_seed = df_train_seed['fuel_efficiency_mpg'].values
    y_val_seed = df_val_seed['fuel_efficiency_mpg'].values

    del df_train_seed['fuel_efficiency_mpg']
    del df_val_seed['fuel_efficiency_mpg']

    X_train_seed = df_train_seed.fillna(0).values
    X_val_seed = df_val_seed.fillna(0).values

    w0, w = train_linear_regression(X_train_seed, y_train_seed)
    y_pred = w0 + X_val_seed @ w

    score = rmse(y_val_seed, y_pred)
    scores.append(score)

    print(seed, round(score, 4))

0 2.2392
1 2.2021
2 2.1634
3 2.2044
4 2.2019
5 2.2458
6 2.2697
7 2.1908
8 2.2166
9 2.207


In [28]:
round(np.std(scores), 3)

np.float64(0.029)

In [29]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

In [30]:
df_train_full = pd.concat([df_train, df_val])

y_train_full = df_train_full['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train_full['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [31]:
X_train_full = df_train_full.fillna(0).values
X_test = df_test.fillna(0).values

In [32]:
w0, w = train_linear_regression_reg(
    X_train_full,
    y_train_full,
    r=0.001
)

y_pred = w0 + X_test @ w

round(rmse(y_test, y_pred), 3)

np.float64(2.236)